# Tier 2 — the stability sensitivity $S = \partial h/\partial P$

$h_g^t = \delta_g^t - \delta^{\mathrm{COI},t}$ is what the corridor bounds. The implicit
function theorem on the dynamic equality block $F(x,P)=0$ gives

$$\frac{dx}{dP} = -F_x^{-1}F_P,\qquad S_P = h_P + h_x\frac{dx}{dP},\qquad
\tilde\gamma^\top S_P = \tilde\gamma^\top h_P + \alpha^\top F_P = -\,\mathrm{dyn}$$

In this formulation $h_P = 0$: $\delta$ is an explicit variable, so the corridor row
contains no $P_g$. The identity therefore reduces to
$\tilde\gamma^\top S_P = \alpha^\top F_P$ — minus the dynamic contribution that
`kkt_Pg_terms.csv` already isolates.

**This notebook only does algebra.** Every derivative comes from the Julia export,
which differentiates the actual JuMP model with MOI's AD. Nothing here reconstructs
$E_g$, $(V,\theta)$, $Y_{red}$ or any Jacobian entry.

In [ ]:
import os
from pathlib import Path
import tomllib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.sparse as sp
from scipy.sparse.linalg import splu

# Same convention as the Tier-1 notebook: KKT_RUN_DIR wins, else this repo's RESULTS.
# A root is resolved to the most recent run that carries a Tier-2 export.
RUN_DIR = Path(os.environ.get("KKT_RUN_DIR", r"..\RESULTS"))
if RUN_DIR.is_dir() and not (RUN_DIR / "run_manifest.toml").exists():
    candidates = sorted(p.parents[1] for p in
                        RUN_DIR.glob("Results - */Sensitivity/sensitivity_meta.toml"))
    if not candidates:
        raise FileNotFoundError(
            f"No run under {RUN_DIR.resolve()} has a Sensitivity/ export. Produce one "
            "with `J_KKT=2 julia --project=TSCOPF runs/run_39bus.jl`.")
    RUN_DIR = candidates[-1]

SENS = RUN_DIR / "Sensitivity"
rd = lambda p: pd.read_csv(p, sep=";")

with open(RUN_DIR / "run_manifest.toml", "rb") as fh:
    M = tomllib.load(fh)
with open(SENS / "sensitivity_meta.toml", "rb") as fh:
    SM = tomllib.load(fh)

# Tier 2 prices SEVERAL channels, not one corridor: the angle corridor, the speed
# corridor, and every other active inequality on a state (discovered x-blocks). A
# run may carry any subset — a Delta-omega-only run has no delta rows at all — so
# every read below is conditional and nothing assumes a channel exists.
def fmt(x, spec=".3e"):
    """Meta values can be the STRING 'NaN'/'Inf' (TOML has no nan literal)."""
    return format(x, spec) if isinstance(x, (int, float)) else str(x)

dyn_cfg = M.get("transient", {}).get("dyn_model", {})
n_h, n_ho = SM["n_h_rows"], SM.get("n_h_omega_rows", 0)

print(f"run dir     {RUN_DIR}")
print(f"case        {M['run']['case']} x lf {M['run']['load_factor']}")
print(f"path        {SM['path']}")
print(f"corridor    delta {'ON ' if n_h else 'off'} tol={M['resolved'].get('delta_tol')} rad"
      f"   |   speed {'ON ' if n_ho else 'off'} "
      f"tol={dyn_cfg.get('Delta_omega_tol_pu')} pu ({dyn_cfg.get('bound_style_Delta_omega')})")
print(f"block       F rows {SM['n_F_rows']}  x cols {SM['n_x_cols']}  square={SM['square']}")
print(f"unmapped    rows {SM['n_unmapped_rows']}  cols {SM['n_unmapped_cols']}")
print(f"channels    h_delta {n_h} rows   h_omega {n_ho} rows   "
      f"x-blocks {SM.get('n_x_block_rows', 0)} rows   P cols {SM['n_P_cols']}")
if SM.get("x_block_families"):
    print(f"            blocks: {', '.join(SM['x_block_families'])}")
print()
print("Julia-side checks")
print(f"  adjoint residual (channels)  {fmt(SM['adjoint_residual_max'])}")
print(f"  adjoint residual (+ blocks)  {fmt(SM.get('adjoint_residual_full_max', float('nan')))}")
print(f"  identity residual  JOINT     {fmt(SM.get('identity_residual_joint', float('nan')))}"
      "   <- solver dual tol x ||dx/dP||; grows with n_x, NOT an error")
if "identity_residual_unexplained" in SM:
    print(f"    of which UNEXPLAINED       {fmt(SM['identity_residual_unexplained'])}"
          "   <- the one that must be ~0")
    print(f"  stationarity residual |r|    {fmt(SM['stationarity_residual_max'])}"
          "   <- equals Ipopt's \"Dual infeasibility\"")
print(f"  identity residual  delta only{fmt(SM['identity_residual']):>12}"
      "   <- non-zero here just means another channel carries the price")
if "fd" in SM:
    print(f"  FD rel err  delta            {fmt(SM['fd']['rel_err'])}")
if "fd_omega" in SM:
    print(f"  FD rel err  omega            {fmt(SM['fd_omega']['rel_err'])}")
print(f"  max |gamma_delta|            {fmt(SM['gamma_binds'])}")
print(f"  max |gamma_omega|            {fmt(SM.get('gamma_omega_binds', 0.0))}")

assert SM["square"], "dynamic block is not square - S is undefined"
if n_h == 0 and n_ho == 0:
    raise RuntimeError("Neither corridor was built - there is nothing to price. "
                       "Set constrain_delta and/or constrain_Delta_omega.")

# Is this run a DISCRIMINATING test, or does the identity hold at zero? A hard assert
# on gamma alone would pass on solver noise (~1e-8), so the test is whether the causer
# charge is material next to the nodal price it is supposed to account for.
def _scale(key):
    v = SM.get(key) or [0.0]
    return max(abs(x) for x in v)

charge_scale = max(_scale("gamma_S_P"), _scale("gamma_S_omega_P"))
if charge_scale < 1e-4:
    print("")
    print("*** NOT A DISCRIMINATING TEST ***")
    print(f"    max |charge| = {charge_scale:.2e} EUR/MW across all channels - no corridor")
    print("    carries a price in this run, so the identity is verified at zero magnitude.")
    print("    It is still a real check of S (see the finite-difference errors above).")
    print("    On the 9-bus Kron case the swing is reshaped through Q and V, which cost")
    print("    nothing, so BOTH corridors go from slack to infeasible without being priced.")
else:
    print("")
    print(f"    DISCRIMINATING: max |charge| = {charge_scale:.3e} EUR/MW.")


In [ ]:
# --- rebuild S from the exported matrices ------------------------------------
# The row/column tables carry a `block` column stamped by the exporter, so the
# partition is not re-declared here - slicing follows the model, not a copy of it.
rows = rd(SENS / "jacobian_rows.csv")
cols = rd(SENS / "jacobian_cols.csv")
coo = rd(SENS / "jacobian_coo.csv")

J = sp.csr_matrix((coo.value, (coo.row - 1, coo.col - 1)),
                  shape=(len(rows), len(cols)))          # Julia is 1-indexed

idx = lambda df, blk, c: df.loc[df.block == blk, c].to_numpy() - 1
F_rows = idx(rows, "F", "row")
x_cols = idx(cols, "x", "col")
P_cols = idx(cols, "P", "col")
P_gen = cols.loc[cols.block == "P", "key1"].to_numpy()

F_x = sp.csc_matrix(J[F_rows][:, x_cols])
F_P = J[F_rows][:, P_cols].toarray()
lu = splu(F_x)
dxdP = -lu.solve(F_P)                       # shared by every channel
alpha = rows.loc[rows.block == "F", "dual"].to_numpy()

# Corridor rows must be ordered the SAME way the exporter ordered them: by declared
# family, then by container key. Taking them in Jacobian row order instead yields a
# row-permuted S. The scalar identity cannot detect that - a sum is invariant to it -
# but every individual sensitivity would be attributed to the wrong (g, t).
def corridor_rows(block, families):
    rank = {f: i for i, f in enumerate(families)}
    sub = rows[rows.block == block].copy()
    sub["rank"] = sub.family.map(rank)
    return sub.sort_values(["rank", "key1", "key2"], kind="stable")

def build_channel(name, blk_up, blk_lo, fams_up, fams_lo):
    """(S, gamma, key, h_P) for one corridor, or None when it has no rows."""
    up = corridor_rows(blk_up, fams_up)
    if up.empty:
        return None
    lo = corridor_rows(blk_lo, fams_lo)
    assert (up[["key1", "key2"]].to_numpy() == lo[["key1", "key2"]].to_numpy()).all(), \
        f"{name}: upper/lower rows do not pair up on (generator, step)"
    r_up = up.row.to_numpy() - 1
    h_x = J[r_up][:, x_cols]
    h_P = J[r_up][:, P_cols].toarray()
    S = h_P + h_x @ dxdP
    gam = up.dual.to_numpy() - lo.dual.to_numpy()
    return dict(name=name, S=S, gam=gam, key=up[["key1", "key2"]].to_numpy(),
                h_P=h_P, rows=r_up)

CH = {}
d = build_channel("delta", "h", "h_lower", SM["families_h"], SM["families_h_lower"])
if d: CH["delta"] = d
w = build_channel("omega", "h_omega", "h_omega_lower",
                  SM.get("families_h_omega", []), SM.get("families_h_omega_lower", []))
if w: CH["omega"] = w

# Every other active inequality on a state. Charged as (mu' b_x) @ dxdP, never
# materialising S_b - the FULL_BUS voltage floor alone is one row per bus per step.
blk = rows[rows.block == "x_block"]
if len(blk):
    b_rows = blk.row.to_numpy() - 1
    bP_blocks = blk.dual.to_numpy() @ J[b_rows][:, P_cols].toarray()
    charge_blocks = (blk.dual.to_numpy() @ J[b_rows][:, x_cols]) @ dxdP + bP_blocks
else:
    bP_blocks = np.zeros(len(P_gen))
    charge_blocks = np.zeros(len(P_gen))

# Stationarity in x from EVERY Jacobian row against its own dual, with no reference to
# the F / h / x_block partition above. Substituting the three S definitions into the
# joint identity and using dxdP = -F_x^-1 F_P collapses the identity residual to exactly
# r' @ dxdP, where r is the classified version of this vector - so the difference between
# the two is the charge of rows the partition missed, and nothing else.
#
# Consequence, and the reason this is computed here rather than read from meta: the raw
# identity residual is the SOLVER's dual tolerance amplified by ||dx/dP||. It grows with
# the state count (~1e-11 at n_x ~ 9e3, ~1e-4 at n_x ~ 2e4 over a 5 s horizon) and is not
# an error. Read `unexplained` instead; it cancels the solver tolerance and stays at 1e-12
# whatever the case size.
r_true = J[:, x_cols].T @ rows.dual.to_numpy()
explained = r_true @ dxdP

# Julia computed the same objects; disagreement means one of the two slicings is wrong.
for nm, fn, c in (("delta", "S_P.csv", CH.get("delta")),
                  ("omega", "S_omega_P.csv", CH.get("omega"))):
    if c is None or not (SENS / fn).exists():
        continue
    jl = rd(SENS / fn).value.to_numpy().reshape(len(c["rows"]), len(P_gen))
    print(f"S_{nm}: python vs julia   max|dS| = {np.abs(c['S'] - jl).max():.3e}"
          f"    max|h_P| = {np.abs(c['h_P']).max():.3e}    rows {len(c['rows'])}")
print(f"F_x: {F_x.shape}, nnz {F_x.nnz}    x-block rows: {len(blk)}")


In [ ]:
# --- the identity: sum of channel charges  vs  -dyn ---------------------------
# gamma' S_P + gamma_omega' S_omega,P + sum_b mu_b' S_b,P
#     = gamma' h_P + gamma_omega' h_omega,P + alpha' F_P + sum_b mu_b' b_P  =  -dyn
charges = {nm: c["gam"] @ c["S"] for nm, c in CH.items()}
charge_total = sum(charges.values()) + charge_blocks

kkt = rd(RUN_DIR / "Dispatch" / "CSV_duals" / "kkt_Pg_terms.csv")

# Which families are dynamic coupling is decided by the exporter, not re-declared here:
# `_PG_FAMILIES` in kkt_pg_terms.jl carries a `store` field (:dyn / :opf) and now writes
# it out per row. Re-deriving that from a list of names is how this cell came to carry a
# `swing_first_step` entry matching nothing the exporter emits - the trapezoidal first
# step is inside `swing_tf`, which is scanned at every step precisely so one entry covers
# both mech_power_modes.
LEGACY_DYN = ["Pe_init", "Pm_init", "gfm_Pmeas_init", "swing_tf", "swing_tpf",
              "corridor_tf_lower", "corridor_tf_upper",
              "corridor_tpf_lower", "corridor_tpf_upper",
              "corridor_ref_tf_lower", "corridor_ref_tf_upper",
              "corridor_ref_tpf_lower", "corridor_ref_tpf_upper",
              "omega_COI_tf_lower", "omega_COI_tf_upper",
              "omega_COI_tpf_lower", "omega_COI_tpf_upper",
              "omega_abs_tf_lower", "omega_abs_tf_upper",
              "omega_abs_tpf_lower", "omega_abs_tpf_upper"]
LEGACY_SCARCITY = ["pg_lower", "pg_upper", "sg_upper", "gfm_Imax", "gfm_Emax"]

if "store" in kkt.columns:
    store = kkt.set_index("family").store.groupby(level=0).first().to_dict()
else:
    # Run exported before the column existed. Same partition, stated by name.
    print("note: kkt_Pg_terms.csv has no `store` column (pre-2026-08-15 export);"
          " falling back to the family-name partition.")
    store = {f: "dyn" for f in LEGACY_DYN}
    store.update({f: "opf" for f in LEGACY_SCARCITY + ["p_balance"]})
    store.update({"cost_gradient": "objective", "RESIDUAL": "check"})

# The partition must be TOTAL. A family in neither bucket used to disappear from `dyn`
# silently and reappear as an identity residual; raising here is the whole point.
unclassified = sorted(set(kkt.family) - set(store))
assert not unclassified, (
    f"families in kkt_Pg_terms.csv with no `store`: {unclassified}. "
    "Add them to _PG_FAMILIES (or to the legacy lists above) - do not let them vanish.")

DYN_FAMILIES = [f for f, s in store.items() if s == "dyn"]
SCARCITY = [f for f, s in store.items() if s == "opf" and f != "p_balance"]

dyn = (kkt[kkt.family.isin(DYN_FAMILIES)]
       .groupby("gen_id").contribution.sum()
       .reindex(P_gen).fillna(0.0).to_numpy())

out = pd.DataFrame(index=pd.Index(P_gen, name="gen"))
for nm, ch in charges.items():
    out[f"charge_{nm}"] = ch
out["charge_blocks"] = charge_blocks
out["charge_total"] = charge_total
# The exporter's `minus_dyn` is gamma'h_P + gamma_omega'h_omega,P + alpha'F_P + mu'b_P.
# Tier 1 supplies the first three; the state boxes are not in _PG_FAMILIES at all, so
# their direct P_g term has to be added here or this column and
# charges_by_channel.csv:minus_dyn are different quantities. It is ~0 on every run
# measured so far (state boxes carry no P_g), which is exactly why omitting it went
# unnoticed - it is not a reason to keep omitting it.
out["-dyn"] = -dyn + bP_blocks
out["residual"] = out["charge_total"] - out["-dyn"]
out["explained"] = explained
out["unexplained"] = out["residual"] - explained
display(out.round(6))
print(f"max |residual|    = {out['residual'].abs().max():.3e}"
      "   <- solver dual tolerance x ||dx/dP||, grows with n_x")
print(f"    explained     = {out['explained'].abs().max():.3e}")
print(f"    UNEXPLAINED   = {out['unexplained'].abs().max():.3e}"
      "   <- the joint identity; this is the one that must be ~0")


In [ ]:
# --- the numbers behind the figures ------------------------------------------
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

for nm, c in CH.items():
    S, gam, key = c["S"], c["gam"], c["key"]
    gens = np.unique(key[:, 0])
    unit = "rad" if nm == "delta" else "pu"
    S_sum = np.vstack([S[key[:, 0] == g].sum(axis=0) for g in gens])
    print(f"(a-{nm}) sum_t S   [{unit} / p.u.]   row = corridor of, column = dispatch of")
    display(pd.DataFrame(S_sum, index=[f"h_G{g}" for g in gens],
                         columns=[f"P_G{i}" for i in P_gen]).round(4))

    bind = np.abs(gam) > 1e-6
    print(f"(b-{nm}) binding corridor rows: {int(bind.sum())} of {len(gam)}")
    if bind.any():
        det = pd.DataFrame({"gen_h": key[bind, 0], "t_index": key[bind, 1],
                            "gamma": gam[bind]})
        for c_, i in enumerate(P_gen):
            det[f"charge_P{i}"] = gam[bind] * S[bind, c_]
        det = det.loc[det.gamma.abs().sort_values(ascending=False).index]
        display(det.head(15).round(6))
    print("")

# Reconciliation against Tier 1, term by term. `dyn_total` is every P_g-carrying row
# of the dynamic block, which is the quantity the identity pairs with.
piv = (kkt[~kkt.family.isin(["RESIDUAL"])]
       .pivot_table(index="gen_id", columns="family", values="contribution",
                    aggfunc="sum").fillna(0.0))
present = lambda names: [c for c in names if c in piv.columns]

rec = pd.DataFrame(index=pd.Index(P_gen, name="gen"))
rec["cost"] = piv.get("cost_gradient", 0.0)
rec["scarcity"] = piv[present(SCARCITY)].sum(axis=1)
rec["dyn_total"] = piv[present(DYN_FAMILIES)].sum(axis=1)
rec["lambda"] = -piv["p_balance"]
for nm, ch in charges.items():
    rec[f"charge_{nm}"] = ch
rec["charge_blocks"] = charge_blocks
rec["dyn + charge"] = rec["dyn_total"] - bP_blocks + charge_total
rec["unexplained"] = rec["dyn + charge"] - explained
rec["lambda check"] = rec["lambda"] - (rec["cost"] + rec["scarcity"] + rec["dyn_total"])

print("(c) Tier 1 decomposition vs Tier 2 charges   [EUR/MW]")
display(rec.round(6))
print(f"    max |dyn_total + charges|          = {rec['dyn + charge'].abs().max():.3e}   <- solver tolerance, NOT an error")
print(f"    of which UNEXPLAINED               = {rec['unexplained'].abs().max():.3e}   <- the identity; must be ~0")
print(f"    max |lambda - (cost+scarcity+dyn)| = {rec['lambda check'].abs().max():.3e}   <- Tier 1 stationarity")

print("")
print("(d) Tier-1 contributions by family   [EUR/MW]")
display(piv.round(6))


In [ ]:
n_gen = len(P_gen)
fig, axs = plt.subplots(len(CH), 3, figsize=(16, 4.4 * len(CH)), squeeze=False)

for r, (nm, c) in enumerate(CH.items()):
    S, gam, key = c["S"], c["gam"], c["key"]
    h_gen, h_t = key[:, 0], key[:, 1]
    gens = np.unique(h_gen)
    unit = "rad" if nm == "delta" else "pu"

    S_sum = np.vstack([S[h_gen == g].sum(axis=0) for g in gens])
    im = axs[r][0].imshow(S_sum, cmap="RdBu_r",
                          vmin=-np.abs(S_sum).max(), vmax=np.abs(S_sum).max())
    axs[r][0].set_xticks(range(n_gen), [f"P {g}" for g in P_gen])
    axs[r][0].set_yticks(range(len(gens)), [f"h {g}" for g in gens])
    axs[r][0].set_title(rf"$\Sigma_t\ S_{{{nm}}}$  [{unit}/p.u.]")
    for (r_, c_), v in np.ndenumerate(S_sum):
        axs[r][0].text(c_, r_, f"{v:.1f}", ha="center", va="center", fontsize=9)
    fig.colorbar(im, ax=axs[r][0], shrink=0.85)

    for g in gens:
        m = h_gen == g
        axs[r][1].plot(h_t[m], np.abs(gam[m]), lw=1.2, label=f"|gamma| {g}")
    axs[r][1].set_xlabel("t index")
    axs[r][1].set_title(f"{nm}: binding pattern")
    axs[r][1].legend(fontsize=8)

    # The identity is  total charge = -dyn,  so those two bars must COINCIDE. Plotting
    # +dyn instead would show them mirrored and look like a sign error in the maths.
    # With more than one channel it is the TOTAL that matches; a single channel's bar
    # is only its share, which is why the total is drawn alongside it.
    #
    # Offsets are HALF the width for a pair and one full width for a triple: the gap
    # between bar centres has to equal the bar width or the group reads as separated
    # series rather than one comparison.
    series = [(rf"charge {nm}", charges[nm])]
    if len(CH) > 1:
        series.append(("charge total", out["charge_total"]))
    series.append(("-dyn (Tier 1)", out["-dyn"]))

    w = 0.8 / len(series)
    base = np.arange(n_gen) - 0.5 * w * (len(series) - 1)
    for k, (lab, vals) in enumerate(series):
        axs[r][2].bar(base + k * w, vals, width=w, label=lab)
    axs[r][2].set_xticks(range(n_gen), [f"G{g}" for g in P_gen])
    axs[r][2].set_ylabel("EUR/MW")
    axs[r][2].set_title("the identity: total charge = -dyn")
    axs[r][2].legend(fontsize=8)

fig.tight_layout()
plt.show()


## Reading it

**The identity is not a coincidence and not an approximation.** It follows from
stationarity in the state, $F_x^\top\alpha + h_x^\top\tilde\gamma = 0$: substituting it
into $\tilde\gamma^\top S_P = \tilde\gamma^\top h_x(-F_x^{-1}F_P)$ collapses the
factorisation and leaves $\alpha^\top F_P$. The residual printed above is therefore a
measure of solver tolerance and float64 conditioning, not of modelling error.

**What it means economically.** The sensitivity-weighted causer-pays charge of each
generator — the natural way to bill units for the stability constraints their
dispatch tightens — is already inside its nodal price, with the correct sign. Charging
$\tilde\gamma^\top S_P$ on top of LMP settlement collects the same money twice.

**When the adjoint block is not zero.** Any *other* inequality on a state variable
(`bound_E`, `bound_δ`, `bound_P_m`) contributes its own dual to that state's column,
and the clean identity picks up an extra term. The per-state breakdown above says
which. The proof run turns those boxes off precisely so the corridor is the only
inequality on the state.

**Sign convention.** $S$ is built from the *upper* corridor rows; the lower rows are
$-h$, which is why $\tilde\gamma = \Gamma^+ - \Gamma^-$ is the right weight and why
only one of the two sides is ever non-zero at a given $(g,t)$.